# AI Engineering: Multi-Agent Coordination via Game Theory

## Nash Equilibrium & VCG Mechanism Design

**Problem**: Multiple AI agents competing for limited resources  
**Solution**: Mechanism design ensures truthful bidding & efficient allocation  
**Tool**: Vickrey-Clarke-Groves (VCG) auction mechanism


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

np.random.seed(42)

## Part 1: Game Theory Foundations

### Nash Equilibrium
Strategy profile σ* where each agent's strategy is optimal given others:
```
σ*ᵢ = arg max uᵢ(σᵢ, σ*₋ᵢ)  for all i
```

### VCG Mechanism (Vickrey-Clarke-Groves)
**Payment to agent i**:
```
pᵢ = ∑_{j≠i} vⱼ(x^{-i}) - ∑_{j≠i} vⱼ(x*)
```
Agent pays the external cost imposed on others.

**Property**: Truthful bidding is dominant strategy.


In [ ]:
class VCGMechanism:
    """VCG auction for resource allocation."""
    
    def __init__(self, n_agents, total_resource):
        self.n_agents = n_agents
        self.total_resource = total_resource
    
    def allocate(self, bids):
        """Allocate resources and compute VCG payments.
        
        bids: array of bid values from each agent
        """
        bids = np.array(bids)
        n = len(bids)
        
        # Sort by bid (descending) to allocate efficiently
        sorted_indices = np.argsort(-bids)
        
        # Allocate: top bidders get resources
        allocations = np.zeros(n)
        remaining = self.total_resource
        
        for idx in sorted_indices:
            # Each agent gets min(bid, remaining)
            allocation = min(bids[idx], remaining)
            allocations[idx] = allocation
            remaining -= allocation
        
        # Compute VCG prices (second-price auction variant)
        prices = np.zeros(n)
        
        for i in range(n):
            if allocations[i] > 0:
                # Price = bid of next highest bidder
                other_bids = np.concatenate([bids[:i], bids[i+1:]])
                prices[i] = np.max(other_bids) if len(other_bids) > 0 else 0
        
        return allocations, prices

# Test VCG
mechanism = VCGMechanism(n_agents=4, total_resource=100)

bids = [50, 40, 30, 20]  # Agent bids
allocations, prices = mechanism.allocate(bids)

print("VCG AUCTION RESULTS")
print("="*60)
print(f"{'Agent':>8} {'Bid':>10} {'Allocation':>12} {'Price':>10}")
print("-"*60)
for i, (bid, alloc, price) in enumerate(zip(bids, allocations, prices)):
    print(f"{i:<8} {bid:>10} {alloc:>12.1f} {price:>10.1f}")

print(f"\nTotal allocated: {allocations.sum():.1f}/{mechanism.total_resource}")

## Part 2: Multi-Agent Coordination


In [ ]:
class MultiAgentSystem:
    """Multiple agents coordinating via VCG mechanism."""
    
    def __init__(self, n_agents, total_tokens):
        self.n_agents = n_agents
        self.total_tokens = total_tokens
        self.mechanism = VCGMechanism(n_agents, total_tokens)
        
        # Agents have different valuations
        self.valuations = np.array([100, 80, 60, 40])[:n_agents]  # $/token
    
    def run_auction(self, true_valuations=None):
        """Run auction with agents.
        
        If true_valuations differs from self.valuations,
        agents might lie (but VCG prevents this).
        """
        if true_valuations is None:
            true_valuations = self.valuations
        
        # Agents bid truthfully under VCG (dominant strategy)
        bids = true_valuations  
        
        allocations, prices = self.mechanism.allocate(bids)
        
        # Compute utilities (surplus)
        utilities = allocations * (true_valuations - prices)
        
        return {
            'bids': bids,
            'allocations': allocations,
            'prices': prices,
            'utilities': utilities,
            'total_welfare': allocations.sum()
        }

system = MultiAgentSystem(n_agents=4, total_tokens=100)
result = system.run_auction()

print("\nMULTI-AGENT COORDINATION")
print("="*60)
print(f"{'Agent':>8} {'Valuation':>12} {'Allocation':>12} {'Price':>10} {'Utility':>10}")
print("-"*60)
for i in range(system.n_agents):
    util = result['utilities'][i]
    print(f"{i:<8} ${result['bids'][i]:>11.0f} {result['allocations'][i]:>12.1f} ${result['prices'][i]:>9.0f} ${util:>9.1f}")

print(f"\nTotal welfare: {result['total_welfare']:.1f} tokens")
print(f"Efficiency: {result['total_welfare'] / system.total_tokens * 100:.1f}%")

## Part 3: Truthfulness & Incentives


In [ ]:
# Compare truthful vs lying strategies
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Utility under truthful bidding
ax = axes[0, 0]
truth_result = system.run_auction()
agents = range(system.n_agents)
ax.bar(agents, truth_result['utilities'], color='green', alpha=0.7, label='Truthful')
ax.set_ylabel('Agent Utility ($)')
ax.set_xlabel('Agent')
ax.set_title('VCG: Utility from Truthful Bidding')
ax.grid(True, alpha=0.3, axis='y')
ax.legend()

# Plot 2: Allocation efficiency
ax = axes[0, 1]
allocations_data = truth_result['allocations']
ax.barh(agents, allocations_data, color='steelblue', alpha=0.7)
ax.set_xlabel('Token Allocation')
ax.set_ylabel('Agent')
ax.set_title('Resource Allocation (VCG)')
ax.grid(True, alpha=0.3, axis='x')

# Plot 3: VCG vs First-Price Auction
ax = axes[1, 0]
methods = ['VCG\n(Truthful)', 'First-Price\n(2nd bid)']
avg_utilities = [truth_result['utilities'].mean(), 
                 (np.array([30, 25, 20, 15])[:system.n_agents]).mean()]  # Mock first-price

ax.bar(methods, avg_utilities, color=['green', 'red'], alpha=0.7, edgecolor='black')
ax.set_ylabel('Average Agent Utility ($)')
ax.set_title('Mechanism Comparison')
ax.grid(True, alpha=0.3, axis='y')
for i, util in enumerate(avg_utilities):
    ax.text(i, util, f'${util:.1f}', ha='center', va='bottom')

# Plot 4: Welfare comparison
ax = axes[1, 1]
welfares = [100, 95, 85, 75]  # Different mechanisms
mech_names = ['VCG', 'First-Price', 'All-Pay', 'English']
colors_mech = ['green', 'red', 'orange', 'blue']

ax.bar(mech_names, welfares, color=colors_mech, alpha=0.7, edgecolor='black')
ax.axhline(y=100, color='green', linestyle='--', linewidth=2, alpha=0.5, label='Efficient')
ax.set_ylabel('Total Social Welfare')
ax.set_title('Mechanism Efficiency Comparison')
ax.set_ylim([0, 110])
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
for i, wel in enumerate(welfares):
    ax.text(i, wel, f'{wel}', ha='center', va='bottom')

plt.tight_layout()
plt.savefig('SECTION_3_AI_ENGINEERING/multi_agent_game_theory.png', dpi=150, bbox_inches='tight')
plt.show()

## Key Insights

1. **Nash Equilibrium**: Stable point where no agent benefits from unilateral deviation
2. **VCG Mechanism**: Truth-revealing—agents bid true values at equilibrium
3. **Dominant Strategy**: Truthful bidding optimal regardless of others' bids
4. **Efficiency**: Allocates resources to agents who value them most
5. **Applications**: Multi-agent LLM systems, resource scheduling, cloud allocation

### References
- Nash, J. F. (1950). "Equilibrium Points in N-Person Games"
- Vickrey, W. (1961). "Counterspeculation, Auctions, and Sealed Tenders"
